# 🧠 POLYDIM — Fine-tuning Camino 1
## Google Colab · T4 GPU / TPU v5e-1

**Objetivo:** entrenar un LLM para operar nativamente en el protocolo POLYDIM sin middleware Python.

**Duración estimada:**
- T4 GPU (15 GB VRAM): ~3–4 horas para modelo 3B
- TPU v5e-1: ~1–2 horas para modelo 3B (más rápido, sin cuantización 4-bit)

**Costo:** $0 (Colab gratuito) | $10/mes (Colab Pro, sin límite de tiempo)

**Antes de empezar:**
1. `Runtime → Change runtime type → T4 GPU` (o `TPU v5e-1`)
2. Aceptar licencia en [huggingface.co/meta-llama](https://huggingface.co/meta-llama) si usás Qwen
3. Ejecutar celdas en orden ↓


## ⚙️ Celda 0 — Detectar hardware (T4 vs TPU)


In [ ]:
import subprocess, sys

# Detectar si hay GPU o TPU disponible
def detect_hardware():
    # Intentar GPU
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"✓ GPU detectada: {name} ({vram:.1f} GB VRAM)")
            return "GPU", name, vram
    except ImportError:
        pass
    # Intentar TPU
    try:
        import torch_xla.core.xla_model as xm
        print(f"✓ TPU detectada: {xm.xla_device()}")
        return "TPU", "v5e-1", 0
    except ImportError:
        pass
    print("⚠️  Sin GPU ni TPU — el entrenamiento será muy lento en CPU")
    return "CPU", "CPU", 0

HW_TYPE, HW_NAME, HW_VRAM = detect_hardware()

# Configuración automática según hardware
if HW_TYPE == "GPU":
    if HW_VRAM >= 40:
        USE_4BIT = False; MODEL_SIZE = "8B"; LORA_R = 64; BATCH = 4
        print("Configuración: A100/H100 → Qwen 2.5 7B, LoRA r=64, batch=4")
    elif HW_VRAM >= 24:
        USE_4BIT = False; MODEL_SIZE = "7B"; LORA_R = 64; BATCH = 4
        print("Configuración: RTX 3090/4090 → Qwen 2.5 7B, LoRA r=64, batch=4")
    else:  # T4 15GB
        USE_4BIT = True;  MODEL_SIZE = "3B"; LORA_R = 32; BATCH = 2
        print("Configuración: T4 → Qwen 2.5 3B 4-bit, LoRA r=32, batch=2")
elif HW_TYPE == "TPU":
    USE_4BIT = False; MODEL_SIZE = "3B"; LORA_R = 32; BATCH = 8
    print("Configuración: TPU v5e-1 → Qwen 2.5 3B, LoRA r=32, batch=8")
else:
    USE_4BIT = True; MODEL_SIZE = "3B"; LORA_R = 16; BATCH = 1
    print("Configuración: CPU → modo test (muy lento)")

print(f"\nUSE_4BIT={USE_4BIT}  MODEL_SIZE={MODEL_SIZE}  LORA_R={LORA_R}  BATCH={BATCH}")


✓ GPU detectada: Tesla T4 (15.6 GB VRAM)
Configuración: T4 → Qwen 2.5 3B 4-bit, LoRA r=32, batch=2

USE_4BIT=True  MODEL_SIZE=3B  LORA_R=32  BATCH=2


## 📦 Celda 1 — Instalación de dependencias


In [ ]:
import subprocess, sys

# Instalación base (GPU y TPU)
pkgs_base = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "datasets>=2.18.0",
    "accelerate>=0.28.0",
    "numpy>=1.24.0",
    "gdown",
]
pkgs_gpu  = ["bitsandbytes>=0.43.0"]   # solo GPU para cuantización 4-bit
pkgs_tpu  = ["torch_xla"]              # solo TPU

print("Instalando dependencias base...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs_base, check=True)

if HW_TYPE == "GPU":
    print("Instalando bitsandbytes (GPU 4-bit)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs_gpu, check=True)
elif HW_TYPE == "TPU":
    print("Instalando torch_xla (TPU)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs_tpu, check=True)

print("\n✓ Todas las dependencias instaladas")


Instalando dependencias base...
Instalando bitsandbytes (GPU 4-bit)...

✓ Todas las dependencias instaladas


## 🔗 Celda 2 — Montar Google Drive y descargar archivos POLYDIM


In [ ]:
import os
import shutil
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')
!ls "/content/drive"

print("------------------------------------------")

# Rutas origen y destino
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/POLYDIM_v1_1/"
!ls -R "/content/drive/MyDrive/POLYDIM_v1_1/"

# Archivos POLYDIM requeridos

#!ls (f"{DRIVE_PROJECT_PATH}")
#input("------------------------------------------")



Mounted at /content/drive
MyDrive
------------------------------------------
/content/drive/MyDrive/POLYDIM_v1_1/:
 backlog
 BACKLOG_V18.json
 BACKLOG_V19.json
 BACKLOG_V20.json
 _backup_20260625_234113
 borrar_skills_obsoletos.py
 canon
'CARPETAS_POLYDIM (1).json'
'CARPETAS_POLYDIM (2) (1).json'
'CARPETAS_POLYDIM (2).json'
'CARPETAS_POLYDIM (3).json'
'CARPETAS_POLYDIM (4).json'
'CARPETAS_POLYDIM (5).json'
 CARPETAS_POLYDIM.json
 cleanup_borrar.py
 colab
 core
 _DEPRECATED
 docs
 _DROPZONE
 evaluacion_proyecto_2026_06_17.md
 guia-instalacion-alumnos.md
 knowledge
 locks
 polydim_camino1
 POLYDIM_DRIVE_OPS_QUEUE.json
 polydim_local_executor.py
 POLYDIM_LOCAL_OPS_ejemplo.json
 POLYDIM_LOCAL_OPS_TASK_CLEANUP_OLD_20260627.json
 POLYDIM_LOCAL_OPS_TASK_CLEANUP_SKILLS_20260627.json
 python_rescate_desastres_claude.py
 relays
 rescatar_desde_old.py
 research
 RESEARCH_DIM_MEMORY_MPAT5.md
 session
 skills
 spec
 TEST_MCP_2026-06-26.md
 tests_rust
 toolbox
 write_paper_complete.py

/content/driv

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/POLYDIM_v1_1/polydim_camino1"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Checkpoints se guardarán en: {OUTPUT_DIR}")
print("\nCopiando archivos del proyecto desde tu Drive montado...")
FILES = [
    "polydim_runtime_v04.py",
    "polydim_weighted_inference.py",
    "polydim_dataset_generator.py"
]

for filename in FILES:
    src_path = os.path.join(DRIVE_PROJECT_PATH, "core", filename)
    print("*"*50)
    print (f"{src_path =}")
    # Comprobar si el archivo existe en Drive
    if not os.path.exists(src_path):
        # Si el v04 no existe, intentar copiar v03 renombrándolo
        if filename == "polydim_runtime_v04.py":
            alternative = os.path.join(DRIVE_PROJECT_PATH, "core", "polydim_runtime_v03.py")
            if os.path.exists(alternative):
                print(f"  -> {filename} no encontrado, copiando y renombrando {alternative}...")
                shutil.copy(alternative, "polydim_runtime_v04.py")
                continue
        print(f"❌ ERROR: No se encontró {src_path} en tu Google Drive.")
        print(f"   Verifica que tu estructura de Drive tenga la carpeta {DRIVE_PROJECT_PATH}")
    else:
        print(f"  Copiando {filename} desde Drive...")
        shutil.copy(src_path, filename)

print("\n✓ Archivos POLYDIM listos en el entorno local de Colab")

Checkpoints se guardarán en: /content/drive/MyDrive/POLYDIM_v1_1/polydim_camino1

Copiando archivos del proyecto desde tu Drive montado...
**************************************************
src_path ='/content/drive/MyDrive/POLYDIM_v1_1/core/polydim_runtime_v04.py'
  Copiando polydim_runtime_v04.py desde Drive...
**************************************************
src_path ='/content/drive/MyDrive/POLYDIM_v1_1/core/polydim_weighted_inference.py'
  Copiando polydim_weighted_inference.py desde Drive...
**************************************************
src_path ='/content/drive/MyDrive/POLYDIM_v1_1/core/polydim_dataset_generator.py'
  Copiando polydim_dataset_generator.py desde Drive...

✓ Archivos POLYDIM listos en el entorno local de Colab


## 📊 Celda 3 — Generar dataset de entrenamiento


In [ ]:
import sys
sys.path.insert(0, '/content')
from polydim_dataset_generator import DatasetGenerator

N_SAMPLES = 5000   # aumentar a 10000 si hay tiempo / Colab Pro

print(f"Generando {N_SAMPLES} pares de entrenamiento...")
gen = DatasetGenerator(seed=42)
samples = gen.generate(
    n=N_SAMPLES,
    output_path="polydim_dataset.jsonl",
    verbose=True,
)
print(f"\nEstadísticas: {gen.stats(samples)}")
print(f"\nEjemplos:")
for s in samples[:3]:
    print(f"  '{s.text[:70]}...' → {s.dominant_dim}")


Generando 5000 pares de entrenamiento...
Generando 5000 pares: 5000 plantilla + 0 LLM
  100/5000 plantillas...
  200/5000 plantillas...
  300/5000 plantillas...
  400/5000 plantillas...
  500/5000 plantillas...
  600/5000 plantillas...
  700/5000 plantillas...
  800/5000 plantillas...
  900/5000 plantillas...
  1000/5000 plantillas...
  1100/5000 plantillas...
  1200/5000 plantillas...
  1300/5000 plantillas...
  1400/5000 plantillas...
  1500/5000 plantillas...
  1600/5000 plantillas...
  1700/5000 plantillas...
  1800/5000 plantillas...
  1900/5000 plantillas...
  2000/5000 plantillas...
  2100/5000 plantillas...
  2200/5000 plantillas...
  2300/5000 plantillas...
  2400/5000 plantillas...
  2500/5000 plantillas...
  2600/5000 plantillas...
  2700/5000 plantillas...
  2800/5000 plantillas...
  2900/5000 plantillas...
  3000/5000 plantillas...
  3100/5000 plantillas...
  3200/5000 plantillas...
  3300/5000 plantillas...
  3400/5000 plantillas...
  3500/5000 plantillas...
  3600/5000 p

## 🏋️ Celda 4 — Fine-tuning

> Esta celda detecta automáticamente si estás en GPU (T4 con 4-bit) o TPU (sin cuantización).


In [ ]:
import torch, json, glob, os
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType

# ── Selección de modelo (Modelos Open-Access Qwen para evitar errores de licencia/gating) ──
if MODEL_SIZE == "3B":
    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
elif MODEL_SIZE == "7B":
    MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
else:
    MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"Modelo: {MODEL_NAME}")
print(f"4-bit: {USE_4BIT}  LoRA r={LORA_R}  batch={BATCH}")

# ── Tokenizer ──────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── Modelo ─────────────────────────────────────────────────────────────────
if USE_4BIT and HW_TYPE == "GPU":
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb, device_map="auto", trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    OPTIM = "paged_adamw_8bit"
    FP16  = True
elif HW_TYPE == "TPU":
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, trust_remote_code=True
    ).to(device)
    OPTIM = "adamw_torch"
    FP16  = False
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )
    OPTIM = "adamw_torch"
    FP16  = True

# ── LoRA ───────────────────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    inference_mode=False,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── Formateo de muestras ───────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Eres un agente POLYDIM. Dado texto en lenguaje natural, identifica "
    "las dimensiones semánticas activas y genera el ObjectND en JSON.\n"
    "Dimensiones: DIM_SQL, DIM_PYTHON, DIM_FLUTTER, DIM_RUST, DIM_GRAPH, "
    "DIM_VECTOR, DIM_TIME, DIM_ERROR, DIM_META\n"
    'Responde SOLO con JSON: {"dims": {...}, "dominant_dim": "DIM_X"}'
)

def fmt(s):
    prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{s['text']}\n<|assistant|>\n"
    target = json.dumps({
        "dims": {
            d: {"w": round(s["activations_gt"].get(d, 0.5), 2), "props": p}
            for d, p in s["dims_declared"].items()
        },
        "dominant_dim": s["dominant_dim"],
    })
    return prompt + target + tokenizer.eos_token

raw = []
with open("polydim_dataset.jsonl") as f:
    for line in f:
        if line.strip(): raw.append(json.loads(line))

n_val = max(1, int(len(raw) * 0.1))
MAX_LEN = 384

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN, padding=False)

ds_tr = Dataset.from_list([{"text": fmt(s)} for s in raw[n_val:]]).map(tok, batched=True, remove_columns=["text"])
ds_va = Dataset.from_list([{"text": fmt(s)} for s in raw[:n_val]]).map(tok, batched=True, remove_columns=["text"])
print(f"Train: {len(ds_tr)}  Val: {len(ds_va)}")

# ── Resume automático desde checkpoint ────────────────────────────────────
ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
resume = ckpts[-1] if ckpts else None
if resume: print(f"Resumiendo desde: {resume}")

# ── Training ───────────────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=max(1, 16 // BATCH),
    learning_rate=2e-4,
    fp16=FP16,
    bf16=(HW_TYPE == "TPU"),
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
    load_best_model_at_end=True,
    optim=OPTIM,
    gradient_checkpointing=(HW_TYPE == "GPU"),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tr,
    eval_dataset=ds_va,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("\n🚀 Iniciando entrenamiento...")
trainer.train(resume_from_checkpoint=resume)
trainer.save_model(f"{OUTPUT_DIR}/final")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"\n✓ Modelo guardado en Drive: {OUTPUT_DIR}/final")


Modelo: Qwen/Qwen2.5-3B-Instruct
4-bit: True  LoRA r=32  batch=2


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train: 4500  Val: 500

🚀 Iniciando entrenamiento...


Step,Training Loss,Validation Loss
200,0.060130,0.060763
400,0.057125,0.057657
600,0.055305,0.055136
800,0.054725,0.054435
846,0.054059,0.054423



✓ Modelo guardado en Drive: /content/drive/MyDrive/POLYDIM_v1_1/polydim_camino1/final


## ✅ Celda 5 — Evaluación del modelo entrenado


In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_BASE = MODEL_NAME
MODEL_LORA = f"{OUTPUT_DIR}/final"

print(f"Cargando modelo fine-tuneado desde: {MODEL_LORA}")
tokenizer_eval = AutoTokenizer.from_pretrained(MODEL_BASE)

# Cargar base con la misma configuración de hardware que Celda 4
if USE_4BIT and HW_TYPE == "GPU":
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_BASE, quantization_config=bnb, device_map="auto", trust_remote_code=True
    )
elif HW_TYPE == "TPU":
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_BASE, torch_dtype=torch.bfloat16, trust_remote_code=True
    ).to(device)
else:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_BASE, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )

model_eval = PeftModel.from_pretrained(base, MODEL_LORA)
model_eval.eval()

# Test set: primeras 100 muestras del dataset
test_samples = []
with open("polydim_dataset.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 100: break
        if line.strip(): test_samples.append(json.loads(line))

# SYSTEM PROMPT CORRECTO (idéntico al de entrenamiento)
SYSTEM_EVAL = (
    "Eres un agente POLYDIM. Dado texto en lenguaje natural, identifica "
    "las dimensiones semánticas activas y genera el ObjectND en JSON.\n"
    "Dimensiones: DIM_SQL, DIM_PYTHON, DIM_FLUTTER, DIM_RUST, DIM_GRAPH, "
    "DIM_VECTOR, DIM_TIME, DIM_ERROR, DIM_META\n"
    'Responde SOLO con JSON: {"dims": {...}, "dominant_dim": "DIM_X"}'
)

correct, jaccard_sum = 0, 0.0

for s in test_samples:
    prompt = f"<|system|>\n{SYSTEM_EVAL}\n<|user|>\n{s['text']}\n<|assistant|>\n"
    inputs = tokenizer_eval(prompt, return_tensors="pt").to(model_eval.device)
    with torch.no_grad():
        out = model_eval.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=False)
    raw = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        pred = json.loads(raw.strip())
        pred_dom  = pred.get("dominant_dim", "")
        pred_dims = set(pred.get("dims", {}).keys())
    except Exception:
        pred_dom, pred_dims = "", set()

    gt_dom  = s["dominant_dim"]
    gt_dims = set(s["dims_declared"].keys())
    if pred_dom == gt_dom: correct += 1
    if pred_dims | gt_dims:
        jaccard_sum += len(pred_dims & gt_dims) / len(pred_dims | gt_dims)

n   = len(test_samples)
acc  = correct / n
jacc = jaccard_sum / n

print(f"\n{'='*45}")
print(f"  RESULTADOS EVALUACIÓN CAMINO 1 — {HW_NAME}")
print(f"{'='*45}")
print(f"  dominant_dim accuracy : {acc:.2%}")
print(f"  dims Jaccard          : {jacc:.4f}")
print(f"  Gate C1 (≥85%)        : {'✓ PASSED' if acc >= 0.85 else '✗ FAILED'}")
print(f"{'='*45}")

if acc >= 0.85:
    print("\n🎉 Camino 1 completado exitosamente.")
    print(f"   Modelo guardado en: {OUTPUT_DIR}/final")
    print("   Próximo paso: actualizar el BACKLOG_VIGENTE en Drive con TASK_046 = TERMINADA")
else:
    print(f"\n⚠️  Accuracy insuficiente ({acc:.1%} < 85%).")
    print("   Opciones: más epochs, más datos, o modelo más grande.")

Cargando modelo fine-tuneado desde: /content/drive/MyDrive/POLYDIM_v1_1/polydim_camino1/final


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


  RESULTADOS EVALUACIÓN CAMINO 1 — Tesla T4
  dominant_dim accuracy : 77.00%
  dims Jaccard          : 0.9800
  Gate C1 (≥85%)        : ✗ FAILED

⚠️  Accuracy insuficiente (77.0% < 85%).
   Opciones: más epochs, más datos, o modelo más grande.


In [ ]:
s = test_samples[0]
prompt = f"<|system|>\n{SYSTEM_EVAL}\n<|user|>\n{s['text']}\n<|assistant|>\n"
inputs = tokenizer_eval(prompt, return_tensors="pt").to(model_eval.device)
with torch.no_grad():
    out = model_eval.generate(**inputs, max_new_tokens=200, do_sample=False)
raw = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("--- PROMPT ---")
print(prompt)
print("--- GENERACIÓN DEL MODELO (RAW) ---")
print(repr(raw))

NameError: name 'test_samples' is not defined

In [ ]:
ejemplos = [
    "Consultar usuarios filtrando por nombre", # Debería ser DIM_SQL
    "Pantalla de login con botón de ingresar",  # Debería ser DIM_FLUTTER
    "Crear una función segura de memoria en Rust", # Debería ser DIM_RUST
    "Error de conexión al servidor timeout_s 10", # Debería ser DIM_ERROR
    "Historial de auditoría versión 1.2 por autor juan" # Debería ser DIM_META
]

for prompt_text in ejemplos:
    prompt = f"<|system|>\n{SYSTEM_EVAL}\n<|user|>\n{prompt_text}\n<|assistant|>\n"
    inputs = tokenizer_eval(prompt, return_tensors="pt").to(model_eval.device)
    with torch.no_grad():
        out = model_eval.generate(**inputs, max_new_tokens=200, do_sample=False)
    raw = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Texto: '{prompt_text}'")
    print(f"Respuesta: {raw.strip()}")
    print("-" * 50)

Texto: 'Consultar usuarios filtrando por nombre'
Respuesta: {"dims": {"datos": {"w": 0.78, "props": {"filtrar", "usuarios", "nombre"}}, "servicio": {"w": 0.75, "props": {"consulta"}}}, "dominant_dim": "datos"}
--------------------------------------------------
Texto: 'Pantalla de login con botón de ingresar'
Respuesta: {"dims": {"usuario": 0.78, "login": 0.75, "botón": 0.72}}, "dominant_dim": "usuario"
--------------------------------------------------
Texto: 'Crear una función segura de memoria en Rust'
Respuesta: {"dims": {"rust": {"w": 0.81, "props": {"tipo": "lang", "framework": "safe"}}, "función": {"w": 0.72, "props": {"tipo": "ent", "nombre": "segura"}}}, "dominant_dim": "rust"}
--------------------------------------------------
Texto: 'Error de conexión al servidor timeout_s 10'
Respuesta: {"dims": {"network": {"w": 0.78, "props": {"error": "timeout_s", "servicio": "conexion"}}}, "dominant_dim": "network"}
--------------------------------------------------
Texto: 'Historial de 